# 01 · Data exploration

Learn what the standardised datasets contain: field definitions, types, missing values, distributions, anomalies and quality reports.

> ⚠️ **These notebooks run on the SYNTHETIC demonstration data.** Every number below is fictional and says nothing about real Catalan grid nodes. To explore real data, change `cfg.mode_paths("demo")` to `cfg.mode_paths("real")` once the real stages have run.

**Rule:** notebooks explore; they never hold the only copy of processing logic. Anything worth keeping moves into `src/bess_catalonia/`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from bess_catalonia.config import load_config
from bess_catalonia.pipeline import run_pipeline

cfg = load_config()  # finds the project root from the notebook's location
paths = cfg.mode_paths("demo")  # SYNTHETIC demonstration outputs (results/demo/)

# Make sure the demo outputs exist (fast no-op when they are up to date).
if not (paths.processed_root / "integrate" / "_stage.json").exists():
    run_pipeline("demo", run_all=True, root=cfg.root, log_level="WARNING")
print("Project root:", cfg.root)
print("Reading:", paths.results_root)

## 1. Field definitions (data contract)
The canonical columns are defined in `bess_catalonia.schemas` and documented in `docs/data_contract.md`.

In [ ]:
from dataclasses import asdict

from bess_catalonia.schemas import SCHEMAS

pd.DataFrame([asdict(f) for f in SCHEMAS["integrated_nodes"]])

## 2. Load the integrated node dataset and check data types

In [ ]:
nodes = pd.read_csv(paths.processed_root / "integrate" / "nodes_integrated.csv")
print(nodes.shape)
nodes.dtypes.to_frame("dtype").T

## 3. Missing values
Missing means *unknown* - it is never silently replaced by zero. Scoring handles it with an explicit policy per indicator.

In [ ]:
indicators = list(cfg.scoring["indicators"])
missing = nodes[indicators].isna().mean().sort_values()
ax = missing.plot.barh(figsize=(6, 3.5), color="#c0392b")
ax.set_xlabel("share of nodes with missing value")
ax.set_title("Missing indicator values (synthetic demo)")
plt.tight_layout()

## 4. Distributions

In [ ]:
axes = nodes[indicators].hist(figsize=(11, 7), bins=12, color="#1f3b73")
plt.suptitle("Indicator distributions (synthetic demo)")
plt.tight_layout()

## 5. Simple anomaly screen
Flags values beyond 1.5 × IQR. An outlier is a question for the data owner, not automatically an error.

In [ ]:
def iqr_outliers(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile([0.25, 0.75])
    span = 1.5 * (q3 - q1)
    return (s < q1 - span) | (s > q3 + span)

flags = {c: nodes.loc[iqr_outliers(nodes[c]), "node_id"].tolist() for c in indicators}
pd.Series(flags, name="outlier nodes").to_frame()

## 6. Quality reports
Every stage writes `results/<mode>/quality/quality_<dataset>.json` (and `.md`). Here: all warnings and errors across stages.

In [ ]:
import json

rows = []
for f in sorted(paths.quality_dir.glob("quality_*.json")):
    rep = json.loads(f.read_text())
    rows += [{"dataset": rep["dataset"], **i} for i in rep["issues"] if i["level"] != "info"]
pd.DataFrame(rows)[["dataset", "level", "check", "message", "count"]]

## 7. Source resolution is preserved: demand by year and sector

In [ ]:
demand = pd.read_csv(paths.processed_root / "demand" / "demand_municipal_long.csv",
                     dtype={"municipality_code": str, "sector_code": str})
pd.pivot_table(demand, index="sector_name", columns="year", values="consumption_mwh",
               aggfunc=lambda s: s.isna().mean()).style.format("{:.0%}").set_caption(
    "Share of suppressed (null) values by sector and year")